# V4.0 — Group 18–20: Full-Window Sweep + Independent 1025-RF CeNN + Tied 128-Radius Iterations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V40_Group18_20_FullWindow_CeNN1025_128_Sweep_Colab.ipynb)

### Test the weakest V3.9 group, not all 28 blocks

The uploaded V3.9 report recorded **cosine 0.7753** for group (18,19,20), while the fully compressed model fell to **39.5% teacher agreement / 35% gold accuracy**. We therefore keep every other original encoder layer unchanged and experiment only with **original full-attention layer 18 + original sliding layers 19 and 20**.

**Your proposed design, expressed precisely:**
1. Original full-attention block 18, including pretrained QKV, RoPE, native SDPA and FFN, uses a **full-attention window sweep** of **128, 256, 384, 512, 768, 1024 total key positions** (not radii). Nonlocal keys are masked for smaller windows. Its pretrained weights stay frozen.
2. A **separate long-range CeNN weight** uses depthwise kernel 9, dilation 128, giving receptive field **1025** (±512 positions) and its own channel mixing, applied once. This is an **RF of 1025 tokens, not 1025 recurrent iterations**.
3. A **local CeNN weight** uses depthwise kernel 9, dilation 32, giving **radius 128** (width 257). This **identical weight** is reused for each local step, normally twice: \`x → local(W) → local(W)\`, with separate scalar gains for steps. Also compare 1 and 3 iterations to determine whether recurrence helps. An optional untied two-step control is included for testing whether weight sharing is the bottleneck.
4. The resulting **Global + CeNN(1025) + Repeated CeNN(±128)** replaces original blocks 18–20. Other 25 pretrained blocks stay original. Compare *with/without long-range branch*, train local steps against teacher 19/20 outputs, and measure whole-model decisions + CUDA latency.

**Important context limitation:** Original V3.9 capped \`MAX_LEN=512\` and \`source_cfg.max_len\` may further cap actual sequences. If every sample has at most 512 tokens, full windows 512/768/1024 can be mathematically identical; this notebook prints their observed equivalence and **does not claim a separate W1024 quality result** unless genuinely longer sequences were evaluated. Optionally enable \`ALLOW_EXTENDED_CONTEXT=True\` and \`MAX_LEN=1024\`; the notebook first checks encoder positional capacity and that longer actual examples exist. Exceeding the original task's trained length is an extrapolation experiment, not the same benchmark.

### Reproducible protocol

- Case-disjoint **TRAIN, PROBE, AUDIT** from dataset TRAIN split; official TEST untouched.
- First compare all six *pretrained full-window* settings **while keeping slides 19–20 original** on PROBE. Identify equivalent windows by observed context length. Keep up to two distinctive windows, always including the unchanged/full setting as a reference.
- Capture original teacher **input to block 18**, outputs after blocks 18, 19 and 20, and the true ModernBERT call kwargs (including both masks). Train CeNN variants on input generated by each candidate full window and original teacher y19/y20 targets. CPU FP16 teacher caches; gradients only through small CeNN modules.
- Initial architecture search over: full window × local recurrence (1/2/3) × separate 1025-RF branch (on/off); rank by **PROBE downstream teacher agreement, gold accuracy and segment delta fidelity**, not raw hidden-state cosine alone.
- Refine the strongest PROBE candidates with additional steps; optional separate-weight (untied) recurrence control. Save a checkpoint after each trial and allow resume.
- Evaluate the selected model on untouched AUDIT **once**, with causal CeNN branch ablations and matched, interleaved native-versus-student CUDA timings. If agreement/accuracy/speed gates fail, label checkpoint *experimental only*. Native teacher remains available unchanged.

**Defaults**: two distinct full windows, 3 local recurrence counts, long-range branch on/off; two-stage training on a T4. This may take substantial Colab GPU time. The pretrained model is loaded only once and never overwritten.

**Acceptance goals (not achieved yet):** teacher agreement ≥95%, gold drop ≤1 percentage point, actual full-model paired speedup >1× and segment cosine ≥0.99. No results are claimed until Colab runs.


In [ ]:
#@title 1. Install pinned V4.0 dependencies and GPU check
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU (T4 or better)."
print("CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0),"Torch",torch.__version__)


In [ ]:
#@title 2. Frozen pretrained Laya and V4.0 experiment configuration
from pathlib import Path
from contextlib import contextmanager
import copy, json, math, random, time, hashlib, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question, full_attention_indices,
)
from tinycenn_lm.decision_training import require_finite

SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
SEED=2040
TARGET=(18,19,20)
MAX_LEN=512 #@param {type:"integer"}
ALLOW_EXTENDED_CONTEXT=False #@param {type:"boolean"}
FULL_WINDOWS=(128,256,384,512,768,1024)
LOCAL_RADIUS=128
LONG_RF=1025
CENN_RANK=96 #@param {type:"integer"}
ITERATION_CANDIDATES=(1,2,3)
LONG_BRANCH_CANDIDATES=(False,True)
TRAIN_CASES=500 #@param {type:"integer"}
DEV_CASES=240 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES=96 #@param {type:"integer"}
PROBE_CACHE_BATCHES=20 #@param {type:"integer"}
SHORT_TRAIN_STEPS=100 #@param {type:"integer"}
REFINE_EXTRA_STEPS=300 #@param {type:"integer"}
TOP_DISTINCT_WINDOWS=2 #@param {type:"integer"}
REFINE_TOP_CANDIDATES=2 #@param {type:"integer"}
RUN_UNTIED_CONTROL=True #@param {type:"boolean"}
WINDOW_SCREEN_DECISIONS=100 #@param {type:"integer"}
CANDIDATE_PROBE_DECISIONS=100 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
CHECK_EVERY=50 #@param {type:"integer"}
LEARNING_RATE=0.0008 #@param {type:"number"}
BENCH_REPEATS=8 #@param {type:"integer"}
RESUME=True #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}
OUTPUT_DIR=Path("/content/Laya_V40_Group18_20_CeNN")
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR=Path("/content/drive/MyDrive/TinyCeNN-LM/V40_Group18_20")
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
assert TARGET==(18,19,20) and 1<=BATCH<=4
assert 1<=TOP_DISTINCT_WINDOWS<=len(FULL_WINDOWS)
assert 1<=REFINE_TOP_CANDIDATES<=12
assert MAX_LEN<=1024 and TRAIN_CACHE_BATCHES>=1 and PROBE_CACHE_BATCHES>=1
assert TRAIN_CASES+DEV_CASES<=1200
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
        "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"
    ]
))
teacher,source_cfg=build_checkpoint_model(source_dir)
teacher.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
FULL_LAYERS=full_attention_indices(teacher)
assert list(FULL_LAYERS)==[0,3,6,9,12,15,18,21,24,27]
assert len(teacher.encoder.layers)==28
assert TARGET[0] in FULL_LAYERS and TARGET[1] not in FULL_LAYERS and TARGET[2] not in FULL_LAYERS
assert int(teacher.encoder.config.local_attention)//2==64
if ALLOW_EXTENDED_CONTEXT:
    limit=int(getattr(teacher.encoder.config,"max_position_embeddings",0))
    if MAX_LEN>limit:
        raise RuntimeError(f"Extended length {MAX_LEN} exceeds encoder position limit {limit}.")
elif MAX_LEN>int(source_cfg.get("max_len",MAX_LEN)):
    raise RuntimeError("Use MAX_LEN at or below source trained length, or explicitly opt into extended context.")
print("Pretrained original layers:",len(teacher.encoder.layers),"target",TARGET)
print("Native slide radius",int(teacher.encoder.config.local_attention)//2)
print("Full window candidates",FULL_WINDOWS,"local CeNN radius",LOCAL_RADIUS,
      "long-range RF",LONG_RF)
print("CUDA:",torch.cuda.get_device_name(0),"AMP:",amp_dtype)
print("Task pretrained max_len",source_cfg.get("max_len"),"requested",MAX_LEN)


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=(MAX_LEN if ALLOW_EXTENDED_CONTEXT else
                         min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN)))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V4.0 teacher helper")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


In [ ]:
#@title 4. Exact ModernBERT full-window modifier: preserve original RoPE, FFN and native SDPA
ORIGINAL_LAYERS=tuple(teacher.encoder.layers)
ORIGINAL_FULL=ORIGINAL_LAYERS[18]
@torch.no_grad()
def observed_window_equivalence(window,observed_max):
    """Keys in a centered total-width W mask have no excluded positions if W>=2*T-1,
    but at T<=W some queries near edges can still have remote keys. We therefore
    never infer equivalence from W>=T alone: use the actual mask comparison.
    """
    T=int(observed_max)
    if T==0:return False
    return torch.equal(make_band(T,window,torch.device("cpu")),
                       torch.ones((T,T),dtype=torch.bool))

def make_band(T,width,device):
    width=int(width)
    left=(width-1)//2
    right=width-1-left
    pos=torch.arange(T,device=device)
    delta=pos[None,:]-pos[:,None]
    return (delta>=-left)&(delta<=right)

class GlobalWindow18(nn.Module):
    """Wrap the FULL original block, modifying only its additive full mask.

    Original sliding_window_mask remains unchanged, and is passed through.
    For the configured full token window W, query i can attend key j iff
    -floor((W-1)/2) <= j-i <= ceil((W-1)/2).
    """
    def __init__(self,original,width):
        super().__init__()
        self.original=original
        self.width=int(width)
    def forward(self,hidden_states,*args,**kwargs):
        attention_mask=kwargs.get("attention_mask")
        T=hidden_states.shape[1]
        if attention_mask is None or attention_mask.ndim!=4:
            raise RuntimeError("Expected encoder-supplied 4D full additive mask.")
        if attention_mask.shape[-2:]!=(T,T):
            raise RuntimeError(f"Incorrect full mask shape {attention_mask.shape}")
        band=make_band(T,self.width,hidden_states.device)
        if attention_mask.dtype==torch.bool:
            modified=attention_mask & band[None,None,:,:]
        else:
            modified=attention_mask.masked_fill(
                ~band[None,None,:,:],torch.finfo(attention_mask.dtype).min)
        # Never change the native sliding mask in this window ablation.
        args_kwargs=dict(kwargs,attention_mask=modified)
        return self.original(hidden_states,*args,**args_kwargs)

@contextmanager
def native_window_only(width):
    assert all(teacher.encoder.layers[i] is ORIGINAL_LAYERS[i]
               for i in range(28)),"Encoder not restored before window screening."
    try:
        teacher.encoder.layers[18]=GlobalWindow18(ORIGINAL_FULL,width)
        yield teacher
    finally:
        teacher.encoder.layers[18]=ORIGINAL_FULL

@torch.inference_mode()
def teacher_probability_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        selected=items[start:min(start+BATCH,limit)]
        b=direct_batch(selected)
        with torch.autocast("cuda",dtype=amp_dtype):
            logits,_=teacher(**b)
        p=torch.softmax(logits.float().masked_fill(
            ~b["marker_mask"].bool(),-1e9),-1).cpu()
        for row,item in zip(p,selected):
            out.append(dict(probs=row,teacher=int(row.argmax()),
                            gold=int(item["gold_index"]),
                            case_id=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def decision_score(items,reference):
    count=len(reference)
    agreements=correct=teacher_correct=0
    js=0.
    for start in range(0,count,BATCH):
        selected=items[start:min(start+BATCH,count)]
        b=direct_batch(selected)
        with torch.autocast("cuda",dtype=amp_dtype):
            logits,_=teacher(**b)
        p=torch.softmax(logits.float().masked_fill(
            ~b["marker_mask"].bool(),-1e9),-1).cpu()
        for j,item in enumerate(selected):
            idx=start+j
            t=reference[idx]["probs"];s=p[j]
            assert t.shape==s.shape
            middle=(t+s)/2
            js+=float(.5*((t*(t.clamp_min(1e-8).log()-
                                middle.clamp_min(1e-8).log())).sum()+
                           (s*(s.clamp_min(1e-8).log()-
                                middle.clamp_min(1e-8).log())).sum()))
            predicted=int(s.argmax())
            agreements+=int(predicted==reference[idx]["teacher"])
            correct+=int(predicted==item["gold_index"])
            teacher_correct+=int(reference[idx]["teacher"]==item["gold_index"])
    return dict(decisions=count,teacher_agreement=agreements/count,
                teacher_gold_accuracy=teacher_correct/count,
                student_gold_accuracy=correct/count,
                gold_accuracy_drop=(teacher_correct-correct)/count,
                mean_js=js/count)

print("Native attention window wrapper configured.")


In [ ]:
#@title 5. Test all full-attention windows from W128 through W1024 on PROBE ONLY
observed_lengths=[len(item["ids"]) for item in probe_items[:WINDOW_SCREEN_DECISIONS]]
assert observed_lengths,"No PROBE examples."
observed_max=max(observed_lengths)
print("Actual PROBE lengths:",min(observed_lengths),np.median(observed_lengths),
      observed_max)
window_identical={}
for W in FULL_WINDOWS:
    band=make_band(observed_max,W,torch.device("cpu"))
    ident=[prior for prior in window_identical
           if torch.equal(band,make_band(observed_max,prior,torch.device("cpu")))]
    window_identical[W]=(ident[0] if ident else W)
print("Equivalence classes at observed maximum T:",
      {str(w):int(v) for w,v in window_identical.items()})
if len(set(window_identical.values()))<len(FULL_WINDOWS):
    print("WARNING: These large full windows have IDENTICAL attention masks "
          "on the observed dataset. They are not independent comparisons.")
probe_teacher=teacher_probability_reference(probe_items,WINDOW_SCREEN_DECISIONS)
assert len(probe_teacher)==WINDOW_SCREEN_DECISIONS
baseline=decision_score(probe_items,probe_teacher)
assert baseline["teacher_agreement"]==1 and baseline["mean_js"]<1e-7
full_window_screen={}
for W in FULL_WINDOWS:
    reference_W=window_identical[W]
    if W!=reference_W:
        full_window_screen[str(W)]=dict(
            **full_window_screen[str(reference_W)],
            equivalent_to=reference_W,independently_measured=False)
        print(f"Full W{W:4d} same mask as W{reference_W}; reuse metrics.")
        continue
    with native_window_only(W):
        score=decision_score(probe_items,probe_teacher)
    full_window_screen[str(W)]=dict(
        quality=score,equivalent_to=W,independently_measured=True)
    print(f"Full W{W:4d}: agreement={score['teacher_agreement']:.4f} "
          f"gold={score['student_gold_accuracy']:.4f}, JS={score['mean_js']:.6f}")

unique=list(dict.fromkeys(window_identical.values()))
unique.sort(key=lambda w:(
    -full_window_screen[str(w)]["quality"]["teacher_agreement"],
    full_window_screen[str(w)]["quality"]["gold_accuracy_drop"],
    full_window_screen[str(w)]["quality"]["mean_js"]
))
chosen_windows=unique[:TOP_DISTINCT_WINDOWS]
# Ensure the original unrestricted behavior is represented as a control.
full_window_native=next(
    (w for w in unique if observed_window_equivalence(w,observed_max)),None
)
if full_window_native is not None and full_window_native not in chosen_windows:
    chosen_windows[-1]=full_window_native
chosen_windows=list(dict.fromkeys(chosen_windows))
print("Selected distinct full windows for CeNN training:",chosen_windows)
print("Selection uses PROBE only, never AUDIT.")


In [ ]:
#@title 6. Teacher 18→19→20 capture and window-conditioned CeNN training cache
# Populated in next commit.


In [ ]:
#@title 7. Separate RF1025 CeNN and tied radius128 recurrent cell
# Populated in next commit.


In [ ]:
#@title 8. Train and checkpoint stage-one candidate grid
# Populated in next commit.


In [ ]:
#@title 9. Refine top candidates and compare optional untied control
# Populated in next commit.


In [ ]:
#@title 10. Whole-model held-out audit and GPU latency
# Populated in next commit.


In [ ]:
#@title 11. Save report, model adapters and plots
# Populated in next commit.
